<a href="https://colab.research.google.com/github/Golanlalong/Bali-Hydrodynamic-Simulation/blob/main/Clip_Sungai%2C_DEM%2C_dan_Titik_Survey.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import glob
import geopandas as gpd
import rasterio
from rasterio.mask import mask

# 1. Mount Drive & Setup Path Input
from google.colab import drive
drive.mount('/content/drive')

PATH_DRIVE = '/content/drive/MyDrive/'
PATH_DAS = os.path.join(PATH_DRIVE, 'DAS Phyton - Tahun 1')
PATH_SUNGAI = os.path.join(PATH_DRIVE, 'Sungai Phyton - Tahun 1')
PATH_DEM_FOLDER = os.path.join(PATH_DRIVE, 'Bali Elevation - Tahun 1')
PATH_SURVEY = os.path.join(PATH_DRIVE, 'Titik Survey Tahun 1')

# Path untuk menyimpan hasil pemisahan
PATH_OUTPUT = os.path.join(PATH_DRIVE, 'Hasil_Pemisahan_DAS')
os.makedirs(PATH_OUTPUT, exist_ok=True)

# 2. Cari File dalam Folder
das_file = glob.glob(os.path.join(PATH_DAS, "*.shp"))[0]
sungai_file = glob.glob(os.path.join(PATH_SUNGAI, "*.shp"))[0]
survey_file = glob.glob(os.path.join(PATH_SURVEY, "*.shp"))[0]
dem_file = glob.glob(os.path.join(PATH_DEM_FOLDER, "*.tif"))[0]  # Mengambil file raster DEM (.tif)

# 3. Load Data
gdf_das = gpd.read_file(das_file)
gdf_sungai = gpd.read_file(sungai_file)
gdf_survey = gpd.read_file(survey_file)

# 4. Samakan Sistem Proyeksi (CRS)
if gdf_sungai.crs != gdf_das.crs:
    gdf_sungai = gdf_sungai.to_crs(gdf_das.crs)
if gdf_survey.crs != gdf_das.crs:
    gdf_survey = gdf_survey.to_crs(gdf_das.crs)

# 5. Iterasi Berdasarkan Setiap 'gridcode' di File DAS
unique_gridcodes = gdf_das['gridcode'].unique()

for gc in unique_gridcodes:
    # Buat folder khusus untuk setiap gridcode
    folder_name = f"DAS_Gridcode_{gc}"
    output_dir = os.path.join(PATH_OUTPUT, folder_name)
    os.makedirs(output_dir, exist_ok=True)

    # Filter Poligon DAS sesuai gridcode saat ini
    das_sub = gdf_das[gdf_das['gridcode'] == gc]

    # Simpan SHP DAS per gridcode
    das_sub.to_file(os.path.join(output_dir, f"DAS_{gc}.shp"))

    # A. Clip Sungai
    sungai_clipped = gpd.clip(gdf_sungai, das_sub)
    if not sungai_clipped.empty:
        sungai_clipped.to_file(os.path.join(output_dir, f"Sungai_{gc}.shp"))

    # B. Clip Titik Survey
    survey_clipped = gpd.clip(gdf_survey, das_sub)
    if not survey_clipped.empty:
        survey_clipped.to_file(os.path.join(output_dir, f"Titik_Survey_{gc}.shp"))

    # C. Clip DEM (Raster)
    with rasterio.open(dem_file) as src:
        # Menyesuaikan CRS DAS ke CRS Raster jika berbeda
        if das_sub.crs != src.crs:
            das_sub_reproj = das_sub.to_crs(src.crs)
        else:
            das_sub_reproj = das_sub

        geoms = das_sub_reproj.geometry.values
        out_image, out_transform = mask(src, geoms, crop=True)
        out_meta = src.meta.copy()

        out_meta.update({
            "driver": "GTiff",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform
        })

        output_dem_path = os.path.join(output_dir, f"DEM_{gc}.tif")
        with rasterio.open(output_dem_path, "w", **out_meta) as dest:
            dest.write(out_image)

    print(f"Selesai memproses DAS Gridcode: {gc}")

print("\nSemua proses pemisahan selesai!")